In [1]:
import sys
!{sys.executable} -m pip install --user transformers peft evaluate accelerate jiwer soundfile librosa
!{sys.executable} -m pip install --user "datasets==2.21.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '/home/jovyan/.local/lib/python3.11/site-packages')

import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [3]:
processor = AutoProcessor.from_pretrained("KBLab/kb-whisper-large")
processor.tokenizer.set_prefix_tokens(language="no", task="transcribe")

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print('bf16:', torch.cuda.is_bf16_supported())

model_kb = AutoModelForSpeechSeq2Seq.from_pretrained(
    "KBLab/kb-whisper-large",
    dtype=torch.float16
).to(device)

Using device: cuda
bf16: True


Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
# Configure LoRA
# Whisper's attention projections are the standard LoRA target for ASR
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,                        # rank — higher = more capacity, more params
    lora_alpha=64,               # scaling factor, commonly set to 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # encoder + decoder attention
    bias="none",
)

model_kb = get_peft_model(model_kb, lora_config)
model_kb.print_trainable_parameters()

trainable params: 15,728,640 || all params: 1,625,607,680 || trainable%: 0.9676


In [6]:
ds_train = load_dataset("NbAiLab/NST", "no-both", split="train")
ds_test  = load_dataset("NbAiLab/NST", "no-both", split="test")

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [7]:
# Data collator
@dataclass
class WhisperDataCollator:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Extract raw audio arrays from the batch
        print(features[0])
        audio_arrays  = [f["audio"]["array"] for f in features]
        sampling_rate = features[0]["audio"]["sampling_rate"]

        # Feature extraction: raw audio → mel spectrogram
        inputs = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=sampling_rate,
            return_tensors="pt",
            padding=True,
        )

        # Tokenise transcriptions
        label_features = self.processor.tokenizer(
            [f["text"] for f in features],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=448,
        )

        # Replace padding token id with -100 so loss ignores it
        labels = label_features.input_ids.masked_fill(
            label_features.attention_mask.ne(1), -100
        )

        # Strip leading BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        inputs["labels"] = labels
        return inputs

data_collator = WhisperDataCollator(processor=processor)

In [8]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./kb-whisper-lora-no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=500,
    max_steps=5000,
    gradient_checkpointing=True,
    bf16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    report_to="none",
)

In [10]:
# Training
trainer = Seq2SeqTrainer(
    model=model_kb,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()

ValueError: No columns in the dataset match the model's forward method signature: (input_features, attention_mask, decoder_input_ids, decoder_attention_mask, encoder_outputs, past_key_values, decoder_inputs_embeds, decoder_position_ids, labels, use_cache, output_attentions, output_hidden_states, return_dict, cache_position, kwargs, label, labels, label_ids). The following columns have been ignored: [lang_code, coding, spc, region_of_youth, number_of_recordings, script, dst, qua, byte_format, type, dos_codepage, board, recording_duration, age, memo, recording_time, recording_date, delimiter, t0, sheet_number, utt, frequency, snd, id, sex, directory, recording_session, character_set, text, pid, version, ansi_codepage, channels, speaker_id, remarks, microphone_position, t2, noi, t1, file, imported_sheet_file, audio, region_of_birth]. Please check the dataset and model. You may need to set `remove_unused_columns=False` in `TrainingArguments`.

In [ ]:
model_kb.save_pretrained("./kb-whisper-lora-no")
processor.save_pretrained("./kb-whisper-lora-no")
print("Saved.")